# CodeAlpha Machine Learning Internship - Task 1
## Production Credit Scoring & Loan Risk Prediction System

This notebook covers the complete end-to-end Machine Learning lifecycle for Credit Risk Assessment:
1. **Data Loading & Generation**: Loading financial applicant records.
2. **Exploratory Data Analysis (EDA)**: Visualizing target balance, credit distributions, and risk correlations.
3. **Feature Engineering & Preprocessing**: DTI calculations, payment ratios, median imputation, one-hot encoding, and scaling.
4. **Model Comparison & Hyperparameter Tuning**: Evaluating 5 ML models (Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, XGBoost) via 5-Fold Stratified Cross-Validation.
5. **Performance Evaluation**: Accuracy, Precision, Recall, F1-Score, ROC-AUC, Confusion Matrix, and ROC curves.
6. **Model Explainability**: Global and local feature importances and SHAP values.
7. **Serialization**: Saving the winning model and preprocessing artifacts.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data.generate_dataset import save_default_dataset
from src.data_loader import load_credit_data
from src.preprocessing import prepare_data_and_preprocessor, save_preprocessor
from src.train import train_and_tune_models, save_best_model
from src.evaluate import generate_evaluation_report, plot_confusion_matrix
from src.explainability import generate_shap_analysis, get_feature_importances

# Set plotting aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
print("[+] All dependencies loaded successfully.")

### Step 1: Load Credit Dataset

In [ ]:
raw_data_path = "../data/raw/credit_data.csv"
if not os.path.exists(raw_data_path):
    save_default_dataset(os.path.dirname(raw_data_path))

df = load_credit_data(raw_data_path)
print(f"Dataset Shape: {df.shape}")
df.head()

### Step 2: Exploratory Data Analysis (EDA)

In [ ]:
# Target variable distribution (Default vs Good Risk)
print("Target Class Proportions:")
print(df['default_status'].value_counts(normalize=True))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x='default_status', data=df, ax=axes[0], palette='Set2')
axes[0].set_title('Loan Default Status Distribution')
axes[0].set_xticklabels(['Good Risk (0)', 'Default (1)'])

sns.boxplot(x='default_status', y='credit_score', data=df, ax=axes[1], palette='Set2')
axes[1].set_title('Credit Score by Default Status')
axes[1].set_xticklabels(['Good Risk (0)', 'Default (1)'])
plt.tight_layout()
plt.show()

### Step 3: Data Preprocessing & Feature Engineering

In [ ]:
X_train, X_test, y_train, y_test, pipeline, feature_names = prepare_data_and_preprocessor(
    df, target_col='default_status', test_size=0.2, random_state=42
)
save_preprocessor(pipeline, feature_names, "../models/preprocessor.joblib")

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Number of Features: {len(feature_names)}")

### Step 4: Train & Tune Machine Learning Models

In [ ]:
trained_results, comparison_df = train_and_tune_models(
    X_train, y_train, cv_folds=5, n_iter=10, scoring='roc_auc'
)
comparison_df

### Step 5: Test Set Evaluation & ROC Curves

In [ ]:
eval_df = generate_evaluation_report(trained_results, X_test, y_test, output_dir="../reports")
eval_df

In [ ]:
best_model_name = eval_df.iloc[0]['Model']
best_model = trained_results[best_model_name]['model']
print(f"Winning Model: {best_model_name}")

# Plot Confusion Matrix for Best Model
y_pred = best_model.predict(X_test)
plot_confusion_matrix(y_test, y_pred, best_model_name, output_path="../reports/confusion_matrix.png")

# Save Best Model
save_best_model(best_model, best_model_name, "../models/best_model.joblib")

### Step 6: Model Explainability & Feature Importances

In [ ]:
generate_shap_analysis(best_model, X_test[:100], feature_names, output_dir="../reports")
df_imp = get_feature_importances(best_model, feature_names)
df_imp.head(10)